Cell 1 : Import Libraries
- 

In [1]:
import pandas as pd 
import numpy as numpy

from sklearn.feature_extraction.text import TfidfVectorizer

Cell 2 : Load Data
-

In [2]:
df = pd.read_csv("../data/processed_jobs.csv")

df.head()

,job_id,job_title,job_description,requirements,benefits,company_name,company_profile,industry,employment_type,location,salary_range,required_experience_years,education_level,department,posting_date,application_deadline,contact_email,company_website,has_logo,num_open_positions,job_function,telecommuting,fraud_reason,text_length,is_fake,combined_text
0,1,Software Engineer,We are looking for responsibilities fast-paced...,Candidates should have dynamic team skills fas...,We offer required skills fast-paced skills req...,Company_543,Our company growth fast-paced responsibilities...,Marketing,Contract,"Toronto, Canada",$60k-$80k,8,High School,Sales,2023-11-24,2024-09-16,hr312@company.com,https://www.company.com,0,3,Management,0,NaN,89,0,software engineerour company growth fast paced...
1,2,Content Writer,We are looking for required support experience...,Candidates should have required team fast-pace...,We offer fast-paced dynamic dynamic strategy g...,Company_192,Our company fast-paced opportunity innovation ...,Finance,Full-time,"Toronto, Canada",$40k-$60k,10,Bachelor,HR,2023-03-03,2024-10-18,hr127@company.com,https://www.company.com,0,10,Development,1,NaN,89,0,content writerour company fast paced opportuni...
2,3,Customer Support Specialist,We are looking for dynamic required fast-paced...,Candidates should have preferred knowledge opp...,We offer skills experience required growth res...,NaN,We are global innovation growth skills knowled...,Healthcare,Internship,Remote,$60k-$80k,3,Bachelor,Marketing,2023-07-31,2024-01-13,job92@gmail.com,NaN,0,6,Support,0,Suspicious email,69,1,customer support specialistwe global innovatio...
3,4,Data Analyst,We are looking for collaboration skills suppor...,Candidates should have innovation team require...,We offer strategy strategy dynamic support opp...,Company_95,Our company fast-paced support team strategy i...,Healthcare,Part-time,"Berlin, Germany",Not Disclosed,10,Bachelor,Sales,2023-08-14,2024-02-09,hr366@company.com,https://www.company.com,1,4,Management,1,NaN,89,0,data analystour company fast paced support tea...
4,5,Graphic Designer,We are looking for team growth growth fast-pac...,Candidates should have experience preferred kn...,We offer opportunity skills responsibilities c...,NaN,We are global experience skills preferred fast...,Retail,Part-time,"London, UK",$60k-$80k,7,High School,Design,2023-04-22,2024-08-26,job359@gmail.com,NaN,0,5,Management,0,No salary info,69,1,graphic designerwe global experience skills pr...


Cell 3 - Check Missing Values

In [3]:
df.isnull().sum()

job_id                          0
job_title                       0
job_description                 0
requirements                    0
benefits                        0
company_name                 1472
company_profile                 0
industry                        0
employment_type                 0
location                        0
salary_range                    0
required_experience_years       0
education_level                 0
department                      0
posting_date                    0
application_deadline            0
contact_email                   0
company_website              1472
has_logo                        0
num_open_positions              0
job_function                    0
telecommuting                   0
fraud_reason                 1528
text_length                     0
is_fake                         0
combined_text                   0
dtype: int64

Cell 4 - Select Structured Features
-

In [4]:
structured_cols = [
    "required_experience_years",
    "num_open_positions",
    "has_logo",
    "telecommuting"
]

df[structured_cols].head()

,required_experience_years,num_open_positions,has_logo,telecommuting
0,8,3,0,0
1,10,10,0,1
2,3,6,0,0
3,10,4,1,1
4,7,5,0,0


Cell 5 - Fill Missing Values
-

In [5]:
df["required_experience_years"] = df["required_experience_years"].fillna(0)

df["num_open_positions"] = df["num_open_positions"].fillna(1)

df["has_logo"] = df["has_logo"].fillna(0)

df["telecommuting"] = df["telecommuting"].fillna(0)

Cell 6 - Convert Yes/No Columns 
-

In [6]:
df["has_logo"].unique()

array([0, 1])

Cell 7 - Scale Numerical Features 
-

In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

numeric_features = scaler.fit_transform(
    df[structured_cols]
)

Cell 8 - TF-IDF
-

In [8]:
tfidf = TfidfVectorizer(
    max_features=6000,
    ngram_range=(1,2),
    stop_words="english"
)

text_features = tfidf.fit_transform(
    df["combined_text"]
)

Cell 9 - Combine Text + Structured Features 
-

In [9]:
from scipy.sparse import hstack 

X = hstack([
    text_features,
    numeric_features
])

y = df["is_fake"]

Cell 10 - Train/Test Split
-

In [10]:
from sklearn.model_selection import train_test_split 

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

Cell 11 - Save Feature Objects
-

In [11]:
import pickle

pickle.dump(
    tfidf,
    open("../models/tfidf.pkl","wb")

)

pickle.dump(
    scaler,
    open("../models/scaler.pkl","wb")
)

In [12]:
print(text_features.shape)
print(numeric_features.shape)
print(X.shape)

(3000, 5480)
(3000, 4)
(3000, 5484)
